# MODIS Aqua daily chl over DBO — **streaming, single projection** into EASE-Grid 2.0 North (4 km)

Storage-free companion to `02_modis-chlor-a-daily-ease2.ipynb`. **Nothing is written to disk.** The notebook
streams each daily MODIS L3m granule, bucket-bins it **once, directly onto EASE-Grid 2.0 North** (DBO
4 km — the same grid as the SSS / wind / PACE notebooks), and folds it into in-memory accumulators.
The per-day array is discarded immediately, so the whole record fits in RAM and uses **no local
storage**.

**One streaming pass builds everything:**
- `box_ts` — per-DBO-box daily mean & median time series (drives all the §6 analyses).
- `mean_map` — running per-pixel time mean (the §7 map).
- `annual_means` — per-year Mar–Sep per-pixel means (the §8 anomaly GIF).

**Trade-off:** every fresh kernel re-streams the full record (slow — hours for ~17k days). There's an
**optional** switch (`CACHE_DERIVED`) to pickle just the small derived products (~80 MB, *not* the
50 GB daily archive) so restarts skip the stream. It's **off by default** to honor "no storage."
Use `STREAM_STRIDE` to sample (e.g. every 7th day) for a fast preview.

## 1. Imports

In [1]:
import time as _time
from pathlib import Path
from datetime import date, timedelta

import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da
import earthaccess
from pyresample import AreaDefinition
from pyresample.bucket import BucketResampler
from matplotlib.path import Path as MplPath
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm, Normalize
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
print("imports OK")

imports OK


## 2. Configuration — record window, EASE-2 DBO 4 km grid, streaming controls

In [2]:
START_DATE = date(2002, 7, 4)     # MODIS Aqua first light
END_DATE   = date.today()
MODIS_SHORT_NAME = "MODISA_L3m_CHL"

LAT_MIN, LAT_MAX = 45, 84          # native MODIS lat band kept before binning
LON_MIN, LON_MAX = -180, -100      # native lon band (Pacific-Arctic) — keeps per-granule memory small

# 1 = stream every available day. Set e.g. 7 to sample ~1 day/week (cuts streaming ~7x) for a preview.
STREAM_STRIDE = 1

# EASE-Grid 2.0 North — DBO 4 km subset (identical to SSS / wind / PACE)
ease2_area = AreaDefinition(
    area_id="ease2_north_dbo_4km",
    description="EASE-Grid 2.0 North subset: Aleutians -> Beaufort (DBO1-8), 4 km",
    proj_id="EPSG:6931",
    projection={"proj":"laea","lat_0":90,"lon_0":0,"x_0":0,"y_0":0,"datum":"WGS84","units":"m"},
    width=906, height=845, area_extent=(-3_624_000, 992_000, 0, 4_372_000),
)

# OPTIONAL tiny cache of derived products only (box_ts + mean_map + annual_means, ~80 MB) — NOT the
# daily archive. Off by default = pure streaming, zero local storage. Flip to True to skip
# re-streaming on kernel restarts.
# Parallel streaming: worker threads each run project_day (I/O-bound) concurrently; the main
# thread folds results serially (no locks). 4 is a safe default; raise toward 8 only if the kernel has plenty of RAM.
N_WORKERS = 4

# Mid-run checkpointing: periodically pickle the accumulators (~160 MB) so a kernel/server death
# RESUMES instead of restarting from day 0. Set False for truly zero disk (but a crash then loses the
# whole multi-hour run). rm the file to force a clean restart.
CHECKPOINT = True
CKPT_PATH  = Path.home() / "modis_ease2_ckpt.pkl"  # add your path
CKPT_EVERY = 500       # save every N newly-streamed days (~12-15 min of work at risk)

CACHE_DERIVED = False
CACHE_PATH    = Path.home() / "modis_ease2_derived.pkl"  # add your path
print("EASE-2 DBO target:", ease2_area.area_id, ease2_area.shape, "| stride:", STREAM_STRIDE,
      "| workers:", N_WORKERS, "| cache:", CACHE_DERIVED)

EASE-2 DBO target: ease2_north_dbo_4km (845, 906) | stride: 1 | workers: 4 | cache: False


## 3. Authenticate + find daily 4 km granules

In [3]:
auth = earthaccess.login(persist=True)

all_granules = earthaccess.search_data(
    short_name=MODIS_SHORT_NAME, temporal=(START_DATE.isoformat(), END_DATE.isoformat()),
)
print(f"{len(all_granules)} total granules in range")

def is_daily_4km(g):
    ur = g["umm"]["GranuleUR"]
    return (((".L3m_DAY_" in ur) or (".L3m.DAY." in ur)) and ((".4km." in ur) or ("_4km." in ur)))

def granule_date(g):
    return date.fromisoformat(g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10])

granules_by_day = {}
for g in all_granules:
    if is_daily_4km(g):
        granules_by_day.setdefault(granule_date(g), g)
if granules_by_day:
    print(f"{len(granules_by_day)} unique daily 4 km days "
          f"({min(granules_by_day)} → {max(granules_by_day)})")
else:
    print("No daily 4 km granules matched — inspect GranuleURs and adjust is_daily_4km().")

22370 total granules in range
8530 unique daily 4 km days (2002-07-04 → 2025-11-30)


## 4. DBO boxes + per-box masks on the EASE-2 grid

In [4]:
# DBO1-8 box corners as literal (lat, lon) — same coordinates as the SSS / wind / PACE notebooks,
# so NO external spreadsheet is needed (works anywhere).
DBO_BOXES = {
    "DBO1": [(63.769462,-172.312422),(61.945988,-172.186927),(61.846516,-175.790978),(63.662718,-176.147111)],
    "DBO2": [(65.107747,-170.865176),(65.097001,-167.859549),(64.481828,-167.907758),(64.493658,-170.840669)],
    "DBO3": [(66.785880,-171.330008),(68.608869,-171.419334),(68.572412,-166.481029),(66.752281,-166.755808)],
    "DBO4": [(71.994000,-164.226000),(71.801000,-159.729000),(70.641000,-160.305000),(70.821000,-164.556000)],
    "DBO5": [(71.807730,-158.537691),(71.635583,-155.931220),(71.110941,-156.307277),(71.278277,-158.847920)],
    "DBO6": [(71.850000,-150.108000),(70.890000,-150.976000),(71.159000,-153.865000),(72.119000,-152.980000)],
    "DBO7": [(70.868000,-141.528000),(69.946000,-142.679000),(70.333000,-145.439000),(71.253000,-144.256000)],
    "DBO8": [(70.834000,-125.155000),(70.049000,-126.997000),(70.664000,-129.354000),(71.442000,-127.431000)],
}
# cartopy / matplotlib paths want (lon, lat)
boxes = {name: [(lon, lat) for (lat, lon) in corners] for name, corners in DBO_BOXES.items()}

lons2d, lats2d = ease2_area.get_lonlats()
grid_pts = np.column_stack([lons2d.ravel(), lats2d.ravel()])
box_masks = {name: MplPath(corners).contains_points(grid_pts).reshape(lons2d.shape)
             for name, corners in boxes.items()}
for name, m in box_masks.items():
    print(f"  {name}: {m.sum():>6,} grid cells inside box")

  DBO1:  2,412 grid cells inside box
  DBO2:    605 grid cells inside box
  DBO3:  2,561 grid cells inside box
  DBO4:  1,293 grid cells inside box
  DBO5:    351 grid cells inside box
  DBO6:    739 grid cells inside box
  DBO7:    773 grid cells inside box
  DBO8:    760 grid cells inside box


## 5. Single-projection day loader (stream → EASE-2, in memory, no write)

In [5]:
def project_day(granule, area):
    """Stream one native MODIS L3m granule and bucket-bin it ONCE onto `area` (EASE-2). Eagerly reads
    the small Pacific-Arctic subset and CLOSES the remote file each call so handles/buffers don't leak
    across thousands of granules (the cause of the earlier OOM)."""
    fobj = earthaccess.open([granule], show_progress=False)[0]
    try:
        with xr.open_dataset(fobj, engine="h5netcdf") as ds:        # eager (no chunks=) -> no lazy file refs
            chl_var = "chlor_a" if "chlor_a" in ds.variables else list(ds.data_vars)[0]
            sub    = ds[chl_var].sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX))
            lon_1d = sub["lon"].values.astype(np.float32)
            lat_1d = sub["lat"].values.astype(np.float32)
            chl_2d = sub.values.astype(np.float32)
    finally:
        try: fobj.close()
        except Exception: pass

    chl_2d = np.where((chl_2d > 0) & np.isfinite(chl_2d), chl_2d, np.nan).astype(np.float32)
    lon2d, lat2d = np.meshgrid(lon_1d, lat_1d)
    resampler = BucketResampler(area, da.from_array(lon2d, chunks="auto"),
                                      da.from_array(lat2d, chunks="auto"))
    ease = resampler.get_average(da.from_array(chl_2d, chunks="auto"))   # *** the single projection ***
    if hasattr(ease, "compute"):
        ease = ease.compute()
    return ease.astype(np.float32)

## 6. The streaming pass (parallel, bounded, checkpointed) — build everything in one sweep

Threads stream + project days; the main thread folds each ~3 MB array into the accumulators and drops
it (flat memory). Each streamed file is closed immediately and `gc` runs periodically, so nothing
leaks across thousands of granules. If `CHECKPOINT=True`, the accumulators are pickled every
`CKPT_EVERY` days — so a kernel/server death **resumes** from the last checkpoint instead of
restarting. Re-running after success is a no-op (products already in memory).

In [6]:
import gc, pickle
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

def _have_products():
    return all(k in globals() and globals()[k] is not None
               for k in ("box_ts", "mean_map", "annual_means"))

ny, nx = ease2_area.shape

# --- restore accumulators from checkpoint, or start fresh ---
if _have_products():
    print("Products already in memory - skipping the stream.")
elif CHECKPOINT and CKPT_PATH.exists():
    with open(CKPT_PATH, "rb") as f:
        _ck = pickle.load(f)
    mean_sum, mean_cnt = _ck["mean_sum"], _ck["mean_cnt"]
    marsep,   box_rows = _ck["marsep"],   _ck["box_rows"]
    print(f"Resumed from checkpoint: {len(box_rows)} days already folded.")
else:
    mean_sum = np.zeros((ny, nx), np.float32); mean_cnt = np.zeros((ny, nx), np.float32)
    marsep   = {}; box_rows = {}

if not _have_products():
    def _stream_one(item):
        d, g = item
        try:
            return d, project_day(g, ease2_area), None
        except Exception as e:
            return d, None, f"{type(e).__name__}: {str(e)[:70]}"

    def _fold(d, arr):
        m = np.isfinite(arr)
        mean_sum[m] += arr[m]; mean_cnt[m] += 1
        if 3 <= d.month <= 9:
            s = marsep.setdefault(d.year, [np.zeros((ny, nx), np.float32),
                                           np.zeros((ny, nx), np.float32)])
            s[0][m] += arr[m]; s[1][m] += 1
        bm, bmed = {}, {}
        for n, mask in box_masks.items():
            v = arr[mask & m]
            bm[n]   = float(np.mean(v))   if v.size else np.nan
            bmed[n] = float(np.median(v)) if v.size else np.nan
        box_rows[d] = (bm, bmed)

    def _save_ckpt():
        if not CHECKPOINT: return
        CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
        tmp = CKPT_PATH.with_suffix(".tmp")
        with open(tmp, "wb") as f:
            pickle.dump({"mean_sum": mean_sum, "mean_cnt": mean_cnt,
                         "marsep": marsep, "box_rows": box_rows}, f)
        tmp.replace(CKPT_PATH)

    all_days  = sorted(granules_by_day.items())[::STREAM_STRIDE]
    remaining = [(d, g) for d, g in all_days if d not in box_rows]   # resume: skip done days
    src  = iter(remaining)
    MAXQ = max(2, N_WORKERS * 2)
    t0 = _time.time(); n_ok = n_fail = since_ck = 0
    print(f"{len(box_rows)} done | streaming {len(remaining)} more with {N_WORKERS} workers…")

    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        inflight = set()
        for _ in range(MAXQ):
            try: inflight.add(ex.submit(_stream_one, next(src)))
            except StopIteration: break
        while inflight:
            done, inflight = wait(inflight, return_when=FIRST_COMPLETED)
            for fut in done:
                d, arr, err = fut.result()
                if arr is None:
                    n_fail += 1
                    if n_fail <= 10: print(f"  {d}  X {err}")
                else:
                    _fold(d, arr); n_ok += 1; since_ck += 1
                    if n_ok % 250 == 0:
                        print(f"  {n_ok} ok / {n_fail} failed  ({(_time.time()-t0)/60:.1f} min)")
                    if since_ck >= CKPT_EVERY:
                        _save_ckpt(); gc.collect(); since_ck = 0
                try: inflight.add(ex.submit(_stream_one, next(src)))
                except StopIteration: pass
            del done
    _save_ckpt()

    # assemble final products from the accumulators
    dates  = sorted(box_rows)
    tcoord = np.array([np.datetime64(d.isoformat()) for d in dates])
    box_ts = {n: {"daily_mean":   xr.DataArray(np.array([box_rows[d][0][n] for d in dates]),
                                               dims="time", coords={"time": tcoord}),
                  "daily_median": xr.DataArray(np.array([box_rows[d][1][n] for d in dates]),
                                               dims="time", coords={"time": tcoord})}
              for n in box_masks}
    mean_map = np.where(mean_cnt > 0, mean_sum / mean_cnt, np.nan)
    _years   = sorted(marsep)
    annual_means = {y: np.where(marsep[y][1] > 0, marsep[y][0] / marsep[y][1], np.nan) for y in _years}
    print(f"\nDone: {len(box_rows)} days folded ({n_fail} failed this pass) "
          f"-> box_ts, mean_map, {len(_years)} annual frames.")

0 done | streaming 8530 more with 4 workers…
  250 ok / 0 failed  (2.8 min)
  500 ok / 0 failed  (5.4 min)
  750 ok / 0 failed  (8.1 min)
  1000 ok / 0 failed  (10.8 min)
  1250 ok / 0 failed  (13.5 min)
  1500 ok / 0 failed  (16.3 min)
  1750 ok / 0 failed  (19.0 min)
  2000 ok / 0 failed  (21.8 min)
  2250 ok / 0 failed  (24.5 min)
  2500 ok / 0 failed  (27.2 min)
  2750 ok / 0 failed  (30.0 min)
  3000 ok / 0 failed  (32.8 min)
  3250 ok / 0 failed  (35.6 min)
  3500 ok / 0 failed  (38.4 min)
  3750 ok / 0 failed  (41.1 min)
  4000 ok / 0 failed  (43.9 min)
  4250 ok / 0 failed  (46.6 min)
  4500 ok / 0 failed  (49.3 min)
  4750 ok / 0 failed  (52.1 min)
  5000 ok / 0 failed  (54.8 min)
  5250 ok / 0 failed  (57.8 min)
  5500 ok / 0 failed  (60.6 min)
  5750 ok / 0 failed  (63.3 min)
  6000 ok / 0 failed  (66.1 min)
  6250 ok / 0 failed  (68.8 min)
  6500 ok / 0 failed  (71.5 min)
  6750 ok / 0 failed  (74.2 min)
  7000 ok / 0 failed  (77.0 min)
  7250 ok / 0 failed  (79.8 min)
  75

/tmp/ipykernel_211/1256632091.py:92: RuntimeWarning: invalid value encountered in divide
  mean_map = np.where(mean_cnt > 0, mean_sum / mean_cnt, np.nan)
/tmp/ipykernel_211/1256632091.py:94: RuntimeWarning: invalid value encountered in divide
  annual_means = {y: np.where(marsep[y][1] > 0, marsep[y][0] / marsep[y][1], np.nan) for y in _years}


## 6b. Save the derived products (box series, mean map, annual means) -> `modis_ease2_products.pkl`

In [20]:
import pickle
from pathlib import Path
PRODUCTS = Path.home() / "modis_ease2_products.pkl"  # add your path
with open(PRODUCTS, "wb") as f:
    pickle.dump({"box_ts": box_ts, "mean_map": mean_map, "annual_means": annual_means}, f)
print(f"Saved {PRODUCTS}  ({PRODUCTS.stat().st_size/1e6:.0f} MB)")


Saved ~/modis_ease2_products.pkl  (80 MB)
